# PDHG Batched Tuning In Colab

This notebook keeps Colab as a thin launcher around the repo's new batched PDHG tuning runner.

It will:
- mount Google Drive
- fetch the repo into `/content`
- install dependencies
- download the FFHQ checkpoint if needed
- create a Colab-specific batched tuning config
- dry-run the candidate grid
- launch the batched sweep
- track live progress through `progress.json`
- show the saved leaderboard


## 1. Runtime

In Colab, go to `Runtime > Change runtime type` and choose:
- `Python 3`
- `GPU`

If available, prefer `A100`. `L4` is also good. Increase `CANDIDATE_CHUNK_SIZE` only after the first dry run looks healthy.

In [ ]:
#@title Project Settings

SETUP_MODE = "git"  #@param ["git", "drive_zip"]
REPO_URL = "https://github.com/Seif-Hussein/dyscode.git"  #@param {type:"string"}
REPO_BRANCH = "main"  #@param {type:"string"}
DRIVE_ZIP_PATH = "/content/drive/MyDrive/mycode2.zip"  #@param {type:"string"}

REPO_DIR = "/content/mycode2"
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/pdhg_tuning_exports"  #@param {type:"string"}

CUSTOM_TUNING_CONFIG = "tuning/pdhg_batched_grid.colab.yaml"

TOTAL_IMAGES = 10  #@param {type:"integer"}
BATCH_SIZE = 10  #@param {type:"integer"}
CANDIDATE_CHUNK_SIZE = 6  #@param {type:"integer"}
MAX_CANDIDATES = 0  #@param {type:"integer"}
PROGRESS_REFRESH_SECONDS = 20  #@param {type:"integer"}


In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title Fetch The Repo
import shutil
import zipfile
from pathlib import Path

repo_dir = Path(REPO_DIR)
if repo_dir.exists():
    shutil.rmtree(repo_dir)

if SETUP_MODE == "git":
    if not REPO_URL:
        raise ValueError("Fill in REPO_URL before running this cell.")
    !git clone --depth 1 --branch "$REPO_BRANCH" "$REPO_URL" "$REPO_DIR"
elif SETUP_MODE == "drive_zip":
    zip_path = Path(DRIVE_ZIP_PATH)
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as handle:
        handle.extractall("/content")
else:
    raise ValueError(f"Unsupported SETUP_MODE: {SETUP_MODE}")

if not repo_dir.exists():
    raise FileNotFoundError(f"Repo directory was not created at {repo_dir}")

%cd /content/mycode2
!git status --short


In [ ]:
#@title Install Python Dependencies
%cd /content/mycode2
!python -m pip install --upgrade pip
!pip install -r requirements.txt


In [ ]:
#@title Download The FFHQ Checkpoint If Needed
%cd /content/mycode2
!mkdir -p pretrained-models
!test -f pretrained-models/ffhq_10m.pt || gdown --id 1BGwhRWUoguF-D8wlZ65tf227gp3cDUDh -O pretrained-models/ffhq_10m.pt
!ls -lh pretrained-models


## 2. Edit The Search Space

This cell writes a separate Colab config at `tuning/pdhg_batched_grid.colab.yaml` so the template stays untouched.

In [ ]:
#@title Build A Colab-Specific Batched Tuning Config
from pathlib import Path
import yaml

repo_dir = Path(REPO_DIR)
template_path = repo_dir / "tuning" / "pdhg_batched_grid.template.yaml"
custom_path = repo_dir / CUSTOM_TUNING_CONFIG

with template_path.open("r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)

cfg["study"]["name"] = "pdhg_phase_retrieval_colab_batched"
cfg["batched"]["candidate_chunk_size"] = int(CANDIDATE_CHUNK_SIZE)
cfg["base_overrides"] = [
    "sampler=edm_pdhg",
    "inverse_task=phase_retrieval",
    "name=PDHG_Batched_Tuning",
    "wandb=false",
    "save_samples=false",
    "save_traj=false",
    "save_traj_raw_data=false",
    "show_config=false",
    f"total_images={int(TOTAL_IMAGES)}",
    f"batch_size={int(BATCH_SIZE)}",
    "num_runs=1",
    "inverse_task.operator.sigma=0.05",
    "sampler.annealing_scheduler_config.num_steps=500",
    "inverse_task.admm_config.denoise.final_step=tweedie",
    "inverse_task.admm_config.max_iter=500",
    "inverse_task.admm_config.dys.gamma=0.0075",
    "inverse_task.admm_config.dys.lambda_schedule.activate=true",
    "inverse_task.admm_config.dys.lambda_schedule.start=1",
    "inverse_task.admm_config.dys.lambda_schedule.end=1",
    "inverse_task.admm_config.dys.lambda_schedule.warmup=0",
    "inverse_task.admm_config.denoise.lgvd.num_steps=0",
]

grid_values = {
    "sampler.annealing_scheduler_config.sigma_max": [20, 27, 30],
    "sampler.annealing_scheduler_config.sigma_min": [0.05, 0.075, 0.1],
    "inverse_task.admm_config.pdhg.tau": [0.005, 0.0075, 0.01],
    "inverse_task.admm_config.pdhg.sigma_dual": [800, 1200, 1600],
}

cfg["grid"] = {
    key: {"values": values}
    for key, values in grid_values.items()
}

with custom_path.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(cfg, handle, sort_keys=False)

print(f"Wrote {custom_path}")
print(custom_path.read_text(encoding='utf-8'))


In [ ]:
#@title Dry Run The Batched Grid
import shlex
import subprocess
import sys
from pathlib import Path

repo_dir = Path(REPO_DIR)
cmd = [
    sys.executable,
    "tuning/run_pdhg_batched_grid.py",
    "--config",
    CUSTOM_TUNING_CONFIG,
    "--dry-run",
]
if MAX_CANDIDATES > 0:
    cmd.extend(["--max-candidates", str(MAX_CANDIDATES)])

print(" ".join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, cwd=repo_dir, check=True)


In [ ]:
#@title Launch The Batched Sweep In Background
import shlex
import subprocess
import sys
from pathlib import Path

repo_dir = Path(REPO_DIR)
cmd = [
    sys.executable,
    "tuning/run_pdhg_batched_grid.py",
    "--config",
    CUSTOM_TUNING_CONFIG,
]
if MAX_CANDIDATES > 0:
    cmd.extend(["--max-candidates", str(MAX_CANDIDATES)])

log_path = repo_dir / "tuning_runs" / "latest_colab_run.log"
log_path.parent.mkdir(parents=True, exist_ok=True)
with log_path.open("w", encoding="utf-8") as handle:
    proc = subprocess.Popen(
        cmd,
        cwd=repo_dir,
        stdout=handle,
        stderr=subprocess.STDOUT,
        text=True,
    )

pid_path = repo_dir / "tuning_runs" / "latest_colab_run.pid"
pid_path.write_text(str(proc.pid), encoding="utf-8")

print("Launched in background:")
print(" ".join(shlex.quote(part) for part in cmd))
print(f"PID: {proc.pid}")
print(f"Log: {log_path}")


In [ ]:
#@title Show Latest Progress
from pathlib import Path
import json
import math

repo_dir = Path(REPO_DIR)
study_root = repo_dir / "tuning_runs" / "pdhg_phase_retrieval_colab_batched"
if not study_root.exists() or not any(study_root.iterdir()):
    raise FileNotFoundError(f"No study runs found at {study_root}")

latest_study = sorted(study_root.iterdir())[-1]
progress_path = latest_study / "progress.json"
if not progress_path.exists():
    raise FileNotFoundError(f"No progress file found at {progress_path}")

progress = json.loads(progress_path.read_text(encoding="utf-8"))
print(f"Study: {latest_study}")
print(f"Status: {progress['status']}")
print(f"Updated: {progress['updated_at']}")
print(
    f"Chunks: {progress['completed_chunks']}/{progress['total_chunks']} | "
    f"Candidates: {progress['completed_candidates']}/{progress['total_candidates']}"
)
print(f"Elapsed: {progress['elapsed_seconds'] / 60:.1f} min")
eta_seconds = progress.get('eta_seconds')
if eta_seconds is None:
    print("ETA: n/a")
else:
    print(f"ETA: {eta_seconds / 60:.1f} min")

best = progress.get("best_so_far")
if best:
    print("Current best:")
    print(
        f"  idx={best['candidate_index']} score={best['score']:.4f} "
        f"sigma_max={best['sigma_max']} sigma_min={best['sigma_min']} "
        f"tau={best['tau']} sigma_dual={best['sigma_dual']}"
    )

log_path = repo_dir / "tuning_runs" / "latest_colab_run.log"
if log_path.exists():
    print("\nRecent log tail:")
    lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    for line in lines[-20:]:
        print(line)


In [ ]:
#@title Launch The Batched Sweep
import shlex
import subprocess
import sys
from pathlib import Path

repo_dir = Path(REPO_DIR)
cmd = [
    sys.executable,
    "tuning/run_pdhg_batched_grid.py",
    "--config",
    CUSTOM_TUNING_CONFIG,
]
if MAX_CANDIDATES > 0:
    cmd.extend(["--max-candidates", str(MAX_CANDIDATES)])

print(" ".join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, cwd=repo_dir, check=True)


In [ ]:
#@title Show The Latest Leaderboard
from pathlib import Path
import pandas as pd

repo_dir = Path(REPO_DIR)
study_root = repo_dir / "tuning_runs" / "pdhg_phase_retrieval_colab_batched"
if not study_root.exists():
    raise FileNotFoundError(f"No study root found at {study_root}")

latest_study = sorted(study_root.iterdir())[-1]
leaderboard = latest_study / "leaderboard.csv"
print(f"Latest study: {latest_study}")
df = pd.read_csv(leaderboard)
display(df)


In [ ]:
#@title Copy The Latest Study Back To Google Drive
from pathlib import Path
import shutil

repo_dir = Path(REPO_DIR)
study_root = repo_dir / "tuning_runs" / "pdhg_phase_retrieval_colab_batched"
latest_study = sorted(study_root.iterdir())[-1]

export_root = Path(DRIVE_EXPORT_DIR)
export_root.mkdir(parents=True, exist_ok=True)
target = export_root / latest_study.name
if target.exists():
    shutil.rmtree(target)
shutil.copytree(latest_study, target)

print(f"Copied study summary to {target}")


## 3. Next Iteration

To adjust the sweep:
- edit `grid_values` in the config-writing cell
- adjust `CANDIDATE_CHUNK_SIZE` if the GPU has more headroom
- rerun the config-writing cell
- rerun the dry-run cell
- rerun the launch cell
